# Detector de residuos con segmentación – NTP 900.058 (7 colores)
Entrena **YOLO11n-seg** con **tus fotos del Drive** (etiquetadas automáticamente) + **TACO**,
en 2 etapas: primero con la base congelada y luego con un ajuste fino.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

### Cómo organizar el Drive
```
MyDrive/Residuos/
├── blanco_plasticos/
│   ├── botella_coca/          ← una subcarpeta por OBJETO (con varias fotos)
│   └── envoltura_galleta/
├── azul_papel_carton/
├── gris_vidrio/
├── amarillo_metales/
├── marron_organicos/
├── rojo_peligrosos/
└── negro_generales/
```
- **Un objeto por foto**, más o menos centrado.
- **Los mismos fondos para todos los colores** (mesa, piso, mano…). Si no, el modelo aprende el fondo y no el material.
- Varía la luz, el ángulo y la distancia. Toma también fotos con la webcam.
- Ideal: 10–20 objetos y 150–300 fotos por color.

## 1. Instalar, descargar el código del proyecto y verificar la GPU

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" pillow-heif "git+https://github.com/ultralytics/CLIP.git"
!rm -rf /content/basurita && git clone -q https://github.com/AlexCoilaJrt/basurita /content/basurita

import sys
sys.path.insert(0, "/content/basurita/src")
import pillow_heif
pillow_heif.register_heif_opener()  # para fotos .HEIC de iPhone

import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Configuración y conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_FOTOS = "/content/drive/MyDrive/Residuos"              # carpetas por color
DRIVE_SALIDA = "/content/drive/MyDrive/Residuos_resultados"  # revision, entrenamientos y best.pt
DIR_TRABAJO = "/content/dataset_seg"                         # dataset final (local, rapido)

USAR_TACO = True
SEMILLA = 42
PROPORCION = (0.7, 0.2, 0.1)  # train / valid / test

import datos_residuos as dr
print("Clases:", dr.CLASES)

## 3. Inventario y división por objeto
Todas las fotos de un mismo objeto (subcarpeta) van al **mismo split**, y se divide **color por color**
para que cada color tenga objetos en train, valid y test.

In [ ]:
import shutil
from collections import Counter

shutil.rmtree(DIR_TRABAJO, ignore_errors=True)
items, avisos = dr.inventario(DRIVE_FOTOS)
avisos += dr.dividir_por_grupos(items, PROPORCION, SEMILLA)

print(f"{'color':<20}{'objetos':>9}{'fotos':>7}   train/valid/test (fotos)")
for c in dr.CLASES:
    sub = [i for i in items if i["clase"] == c]
    por_split = Counter(i["split"] for i in sub)
    print(f"{c:<20}{len({i['grupo'] for i in sub}):>9}{len(sub):>7}   "
          f"{por_split['train']}/{por_split['valid']}/{por_split['test']}")
print(f"\nTotal: {len(items)} fotos")
for a in avisos:
    print("AVISO:", a)

## 4. Etiquetado automático (YOLO-World + SAM)
- **YOLO-World** ubica el objeto, **SAM** dibuja su contorno y **la carpeta** da el color.
- Si YOLO-World no encuentra nada, SAM recorta lo que esté **al centro** (estado `por_centro`).
- Estados a revisar: `por_centro`, `varios_objetos`, `objeto_pequeno`, `solo_caja`, `sin_objeto`.

Tarda ~0.3 s por foto con GPU.

In [ ]:
from ultralytics import YOLOWorld, SAM

modelo_world = YOLOWorld("yolov8s-worldv2.pt")
modelo_sam = SAM("sam2.1_b.pt")
registros = dr.etiquetar_automatico(items, DIR_TRABAJO, modelo_world, modelo_sam)

estados = Counter(r["estado"] for r in registros)
print("Estados:", dict(estados))
dudosos = [r for r in registros if r["estado"] != "ok"]
with open(f"{DIR_TRABAJO}/dudosos.txt", "w") as f:
    for r in dudosos:
        f.write(f"{r['estado']}\t{r['ruta']}\n")
print(f"{len(dudosos)} fotos a revisar (lista en {DIR_TRABAJO}/dudosos.txt)")

## 5. Revisar las etiquetas
Se guardan hojas de miniaturas en `Residuos_resultados/revision/` (Drive). Los **dudosos salen primero, con borde rojo**.

Si una etiqueta está mal (el contorno no cubre el objeto, marcó la mano, etc.):
1. **Borra esa foto del Drive** (o muévela fuera de `Residuos/`).
2. Vuelve a correr desde el **paso 3**.

In [ ]:
from IPython.display import Image as IPImage, display

hojas = dr.galeria_revision(registros, f"{DRIVE_SALIDA}/revision")
print(len(hojas), "hojas guardadas en", f"{DRIVE_SALIDA}/revision")
for c in dr.CLASES:  # muestra la primera hoja de cada color
    primera = [h for h in hojas if f"/{c}_01" in h]
    if primera:
        print(c)
        display(IPImage(primera[0], width=900))

## 6. (Opcional) Agregar TACO
~1 500 fotos de basura real con contornos. La descarga tarda 15–30 min y algunos enlaces de Flickr ya no existen.
Se divide **por batch** (fotos de la misma sesión juntas).
Si vuelves a correr el paso 3, corre esta celda otra vez (el paso 3 limpia el dataset).

In [ ]:
import os
if USAR_TACO:
    if not os.path.isdir("/content/TACO"):
        !git clone -q https://github.com/pedropro/TACO /content/TACO
    !cd /content/TACO && python download.py
    stats, sin_mapeo = dr.convertir_taco("/content/TACO", DIR_TRABAJO, PROPORCION, SEMILLA)
    print("TACO:", stats)
    if sin_mapeo:
        print("Categorias de TACO sin color asignado (se descartan):", sin_mapeo)

## 7. Resumen del dataset final

In [ ]:
tabla, avisos = dr.resumen(DIR_TRABAJO)
print(tabla)
for a in avisos:
    print("AVISO:", a)
YAML = dr.escribir_yaml(DIR_TRABAJO)
print("\n" + open(YAML).read())

## 8. Etapa 1 – base congelada
Parte de `yolo11n-seg.pt` (ya sabe "ver" objetos) y **congela las 10 primeras capas** (backbone):
solo aprende a decidir los 7 colores, sin olvidar lo general.
Los resultados se guardan en el Drive para no perderlos si Colab se desconecta.

In [ ]:
from ultralytics import YOLO

RUNS = f"{DRIVE_SALIDA}/runs"
modelo = YOLO("yolo11n-seg.pt")
modelo.train(
    data=YAML,
    epochs=40,
    imgsz=640,
    batch=16,
    freeze=10,
    patience=15,
    degrees=10,
    seed=SEMILLA,
    project=RUNS,
    name="etapa1_congelada",
    exist_ok=True,
)

## 9. Etapa 2 – ajuste fino (todo descongelado, aprendizaje muy bajo)
Adapta un poco la base a la basura (p. ej. textura de plástico vs. papel) sin perder lo aprendido.

In [ ]:
modelo = YOLO(f"{RUNS}/etapa1_congelada/weights/best.pt")
modelo.train(
    data=YAML,
    epochs=25,
    imgsz=640,
    batch=16,
    optimizer="SGD",   # con optimizer="auto" se ignora lr0
    lr0=0.001,
    lrf=0.1,
    warmup_epochs=0,
    patience=10,
    degrees=10,
    seed=SEMILLA,
    project=RUNS,
    name="etapa2_ajuste_fino",
    exist_ok=True,
)

In [ ]:
# (Solo si Colab se desconecto) Reanudar la etapa que se corto:
# YOLO(f"{RUNS}/etapa1_congelada/weights/last.pt").train(resume=True)
# YOLO(f"{RUNS}/etapa2_ajuste_fino/weights/last.pt").train(resume=True)

## 10. Elegir el mejor modelo (con **valid**) y evaluarlo en **test**
Se elige con valid y se reporta test una sola vez: así la métrica de test es honesta.

In [ ]:
import os
from IPython.display import Image as IPImage, display

candidatos = {e: f"{RUNS}/{e}/weights/best.pt" for e in ("etapa1_congelada", "etapa2_ajuste_fino")}
puntaje = {}
for etapa, ruta in candidatos.items():
    if os.path.exists(ruta):
        m = YOLO(ruta).val(data=YAML, split="val", plots=False, verbose=False)
        puntaje[etapa] = m.seg.map
        print(f"{etapa:<20} valid mAP50-95 (mascaras): {m.seg.map:.3f}")
GANADOR = max(puntaje, key=puntaje.get)
print("\nMejor:", GANADOR)

split = "test" if "test:" in open(YAML).read() else "val"
m = YOLO(candidatos[GANADOR]).val(data=YAML, split=split, project=RUNS, name="evaluacion", exist_ok=True)
print(f"\n{split}: mascaras mAP50 {m.seg.map50:.3f} | mAP50-95 {m.seg.map:.3f} | cajas mAP50 {m.box.map50:.3f}")
indices = list(m.seg.ap_class_index)
for i, c in enumerate(dr.CLASES):
    if i in indices:
        print(f"  {c:<20} AP50 mascara = {m.seg.ap50[indices.index(i)]:.3f}")
    else:
        print(f"  {c:<20} (sin ejemplos en {split})")

In [ ]:
for f in ["confusion_matrix_normalized.png", "MaskPR_curve.png"]:
    ruta = f"{RUNS}/evaluacion/{f}"
    if os.path.exists(ruta):
        print(f)
        display(IPImage(ruta, width=800))
ruta = f"{RUNS}/{GANADOR}/results.png"
if os.path.exists(ruta):
    display(IPImage(ruta, width=900))

## 11. Probar con una foto tuya

In [ ]:
from google.colab import files
mejor = YOLO(candidatos[GANADOR])
for nombre in files.upload():
    r = mejor.predict(nombre, conf=0.4)[0]
    r.save(filename=f"pred_{nombre}")
    display(IPImage(f"pred_{nombre}", width=600))

## 12. Guardar y descargar `best.pt`
Cópialo a `modelos/best.pt` en tu Mac y el detector lo usará automáticamente (con contornos de color).

In [ ]:
shutil.copy(candidatos[GANADOR], f"{DRIVE_SALIDA}/best.pt")
shutil.copy(candidatos[GANADOR], "/content/best.pt")
print("Copia en el Drive:", f"{DRIVE_SALIDA}/best.pt")
files.download("/content/best.pt")